<a href="https://colab.research.google.com/github/Charvi-M/BERTImp/blob/main/BERTimplementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from scipy.spatial.distance import cosine

In [ ]:
"""Now we load the tokenizer to break down sentences into tokens so that
the BERT model uses them for processing."""
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
#we are loading a pretrained BERT tokenizer with the configuration of
#bert-base-uncased, and assigning it to the variable tokenizer
#BertTokenizer: A class from Hugging Face's transformers library that handles
# tokenizing text for BERT models.
#.from_pretrained('bert-base-uncased'):This loads the pretrained tokenizer
#weights and vocabulary from the bert-base-uncased model,
#bert-base: means 12-layer (base) version of BERT.
#uncased: all text will be converted to lowercase, and it ignores case diffrence

In [ ]:
txt = "I need to visit the financial bank tomorrow; after that, we'll set up a"
txt+="tent by the river bank, just across from the bank building"
txt+="where my friend works."

In [ ]:
"""
Now we need to add two tokens called cls and sep in the begining and end
of txt, respectively.
"""
wrangled_txt = '[CLS] ' + txt + ' [SEP]'

#tokenization
tokenized_txt = tokenizer.tokenize(wrangled_txt)

print(tokenized_txt)

In [ ]:
#Next, we convert each token to its corresponding index in BERT’s vocabulary.
#get the ids of the tokens
ids_tokens = tokenizer.convert_tokens_to_ids(tokenized_txt)

#Display the tokens
for t in zip(tokenized_txt, ids_tokens):
    print('{:<12} {:>8,}'.format(t[0], t[1]))

In [ ]:
#For each token in tokenized_text, we need to indicate whether it belongs to
#the first sentence (represented by 0s) or the second (represented by 1s).
#because we have only one sentence we only need a vector of 1s,
# marking all tokens as part of a single sentence.
segments_ids = [1] * len(tokenized_txt)
#Convert the token IDs and segment IDs into tensors.
#This step because BERT expects PyTorch tensors as input not lists or arrays.
token_tensor = torch.tensor([ids_tokens])
segment_tensor = torch.tensor([segments_ids])

In [ ]:
#Loading model from hugging face with the weights
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True, return_dict = True)
# Put the model in "evaluation" mode, meaning feed-forward operation.
model.eval()
#BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True, return_dict = True)
#Loads a pre-trained BERT model (12-layer, 110M parameters) from Hugging Face’s model hub.
#Includes learned weights trained on English Wikipedia + BooksCorpus.
#Enables access to hidden states from all layers, not just the final output.
# output_hidden_states=True: Tells BERT to return the hidden states from all 13
# layers (embedding layer + 12 transformer layers).
#Each layer gives a tensor of shape: (batch_size, sequence_length, hidden_size).


In [ ]:
#Compute the output
with torch.no_grad():
    outputs = model(token_tensor, segment_tensor)

hidden_states = outputs.hidden_states

In [ ]:
#Examining Outputs in detail
#The first one is initial embeddings
print ("Number of layers:", len(hidden_states))
layer_ptr = 0

print ("Number of batches:", len(hidden_states[layer_ptr]))
batch_ptr = 0

print ("Number of tokens:", len(hidden_states[layer_ptr][batch_ptr]))
token_ptr = 0

print ("Number of hidden units:", len(hidden_states[layer_ptr][batch_ptr][token_ptr]))

In [ ]:
#Concatenate all the layers
token_embeddings = torch.stack(hidden_states, dim=0)

#remove the batch dimension
token_embeddings = torch.squeeze(token_embeddings, dim=1)
print(token_embeddings.shape)

In [ ]:
#Concatenate all the layers
token_embeddings = torch.stack(hidden_states, dim=0)

#remove the batch dimension
token_embeddings = torch.squeeze(token_embeddings, dim=1)
print(token_embeddings.shape)

#We now have a single tensor that gives us:

#13 representations for each token, showing how BERT processes it across layers.

#We can do things like:

#Average the last 4 layers (a common practice for sentence embeddings)

#Visualize how a token's meaning evolves through layers

#Extract features from a specific layer

In [ ]:
#Swap the dimensions so that the word embeddings generated from the layers are grouped together.
# Swap dimensions 0 and 1 so that each word contains the 13 layer hidden states
token_embeddings = token_embeddings.permute(1,0,2)

token_embeddings.size()

In [ ]:
#Creating word vectors from the hidden states by summing the embeddings of the last four layers.
#sum the last four layers
token_vectors_sum = []

# token_embeddings is a [35 x 13 x 768] tensor.

# For each token in the sentence...
for token in token_embeddings:

    # `token` is a [12 x 768] tensor

    # Sum the vectors from the last four layers.
    sum_vector = torch.sum(token[-4:], dim=0)

    # Use `sum_vec` to represent `token`.
    token_vectors_sum.append(sum_vector)

print ('Shape is: %d x %d' % (len(token_vectors_sum), len(token_vectors_sum[0])))

In [ ]:
#Displaying the index of the word, as we need it to compare the similarity.
for i, t in enumerate(tokenized_txt):
  print (i, t)

In [ ]:
#compare the word bank in 7, 23, and 29
#txt = "I need to visit the financial bank tomorrow; after that, we'll set up a tent by the river bank, just across from the bank building where my friend works."

same_bank_word = 1 - cosine(token_vectors_sum[7], token_vectors_sum[29])
diff_bank_word1 = 1 - cosine(token_vectors_sum[7], token_vectors_sum[23])
diff_bank_word2 = 1 - cosine(token_vectors_sum[23], token_vectors_sum[29])

print('Vector similarity for  *similar*  meanings:  %.2f' % same_bank_word)
print('Vector similarity for *different* meanings:  %.2f' % diff_bank_word1)
print('Vector similarity for *different* meanings:  %.2f' % diff_bank_word2)